## Animal Classification with PyTorch

In [2]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

In [4]:
!unzip -q "/content/dataset.zip" -d /content/dataset
data_dir = '/content/dataset'

In [5]:
!ls /content/

dataset  dataset.zip  drive  sample_data


In [2]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Transforms
# Training transforms include augmentation to reduce overfitting
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),          # ResNet expects 224x224 input
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet stats —
                          std=[0.229, 0.224, 0.225])    # required since we use pretrained weights
])

# Validation/test transforms: no augmentation, just resize + normalize
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

In [7]:
!ls /content/dataset/dataset

Butterfly  Cat	Chicken  Cow  Dog  Elephant  Horse  Sheep  Spider  Squirrel


In [8]:
# Load dataset and split
data_dir = '/content/dataset/dataset'  # path to your folder

full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f'Classes: {class_names}')

# Split into train/val (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Val set should use val_transform, not train_transform (no augmentation)
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

Classes: ['Butterfly', 'Cat', 'Chicken', 'Cow', 'Dog', 'Elephant', 'Horse', 'Sheep', 'Spider', 'Squirrel']


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print(torch.cuda.is_available())

cuda
True


In [10]:
# Model — transfer learning with ResNet18
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 177MB/s]


In [11]:
# Train
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    for epoch in range(epochs):
        # --- Training phase ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # --- Validation phase ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{epochs} | '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

Epoch 1/10 | Train Loss: 0.5573, Train Acc: 0.8568 | Val Loss: 0.2354, Val Acc: 0.9325
Epoch 2/10 | Train Loss: 0.2344, Train Acc: 0.9328 | Val Loss: 0.1828, Val Acc: 0.9461
Epoch 3/10 | Train Loss: 0.1938, Train Acc: 0.9445 | Val Loss: 0.1665, Val Acc: 0.9493
Epoch 4/10 | Train Loss: 0.1738, Train Acc: 0.9476 | Val Loss: 0.1645, Val Acc: 0.9488
Epoch 5/10 | Train Loss: 0.1591, Train Acc: 0.9517 | Val Loss: 0.1663, Val Acc: 0.9525
Epoch 6/10 | Train Loss: 0.1557, Train Acc: 0.9527 | Val Loss: 0.1569, Val Acc: 0.9525
Epoch 7/10 | Train Loss: 0.1455, Train Acc: 0.9546 | Val Loss: 0.1581, Val Acc: 0.9516
Epoch 8/10 | Train Loss: 0.1315, Train Acc: 0.9585 | Val Loss: 0.1573, Val Acc: 0.9520
Epoch 9/10 | Train Loss: 0.1307, Train Acc: 0.9581 | Val Loss: 0.1571, Val Acc: 0.9498
Epoch 10/10 | Train Loss: 0.1277, Train Acc: 0.9579 | Val Loss: 0.1575, Val Acc: 0.9529


In [12]:
# unfreeze layer4 and fine-tune
# Unfreeze the last convolutional block so it can adapt to your specific classes
for param in model.layer4.parameters():
    param.requires_grad = True

# Two learning rates: small for the pretrained layer4 (don't destroy its features),
# normal for the head which still needs to keep adjusting
optimizer = optim.Adam([
    {'params': model.fc.parameters(), 'lr': 1e-3},
    {'params': model.layer4.parameters(), 'lr': 1e-5}
])

train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

Epoch 1/10 | Train Loss: 0.1284, Train Acc: 0.9574 | Val Loss: 0.1514, Val Acc: 0.9552
Epoch 2/10 | Train Loss: 0.0813, Train Acc: 0.9737 | Val Loss: 0.1519, Val Acc: 0.9556
Epoch 3/10 | Train Loss: 0.0533, Train Acc: 0.9838 | Val Loss: 0.1491, Val Acc: 0.9552
Epoch 4/10 | Train Loss: 0.0362, Train Acc: 0.9915 | Val Loss: 0.1455, Val Acc: 0.9593
Epoch 5/10 | Train Loss: 0.0286, Train Acc: 0.9918 | Val Loss: 0.1477, Val Acc: 0.9547
Epoch 6/10 | Train Loss: 0.0170, Train Acc: 0.9972 | Val Loss: 0.1481, Val Acc: 0.9593
Epoch 7/10 | Train Loss: 0.0151, Train Acc: 0.9971 | Val Loss: 0.1449, Val Acc: 0.9602
Epoch 8/10 | Train Loss: 0.0133, Train Acc: 0.9973 | Val Loss: 0.1628, Val Acc: 0.9570
Epoch 9/10 | Train Loss: 0.0160, Train Acc: 0.9955 | Val Loss: 0.1581, Val Acc: 0.9593
Epoch 10/10 | Train Loss: 0.0081, Train Acc: 0.9986 | Val Loss: 0.1612, Val Acc: 0.9602


In [13]:
# Confusion matrix — check if horse/elephant confusion actually improved
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(classification_report(all_labels, all_preds, target_names=class_names))

              precision    recall  f1-score   support

   Butterfly       0.94      0.97      0.95       199
         Cat       0.97      0.99      0.98       272
     Chicken       1.00      0.96      0.98       200
         Cow       0.89      0.94      0.91       193
         Dog       0.95      0.98      0.97       345
    Elephant       0.97      0.95      0.96       203
       Horse       0.98      0.92      0.94       215
       Sheep       0.95      0.94      0.94       188
      Spider       0.97      0.95      0.96       196
    Squirrel       0.99      0.97      0.98       198

    accuracy                           0.96      2209
   macro avg       0.96      0.96      0.96      2209
weighted avg       0.96      0.96      0.96      2209



In [14]:
# Save the model
torch.save(model.state_dict(), 'animal_classifier_resnet18.pth')

In [15]:
#Inference on a new image
from PIL import Image

def predict_image(image_path, model, transform, class_names):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)  # add batch dimension

    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)

    return class_names[predicted.item()]

# Provide a valid path to an image from your dataset
# For example, using one of the butterfly images that was unzipped earlier
image_to_predict_path = '/content/dataset/dataset/Butterfly/OIP--gEvtip9-FSfOy_MWMXSxQHaJ6.jpeg'
result = predict_image(image_to_predict_path, model, val_transform, class_names)
print(f'Predicted class: {result}')

Predicted class: Butterfly


In [23]:
import os

directory_to_check = '/content/dataset/dataset/Butterfly/'
if os.path.isdir(directory_to_check):
    print(f'The directory "{directory_to_check}" exists.')
    # Optionally, list some files in the directory to confirm it's not empty
    print('Files in the directory (first 5):')
    print(os.listdir(directory_to_check)[:5])
else:
    print(f'The directory "{directory_to_check}" does NOT exist.')

The directory "/content/dataset/dataset/Butterfly/" exists.
Files in the directory (first 5):
['OIP-qugefs9Tnt250B8cygtBJQAAAA.jpeg', 'OIP-WTFhQqD88fBJqFbfVlHSEwHaFa.jpeg', 'OIP-Y5_dpYEmm7Gar2Dyn_EZcAHaHa.jpeg', 'OIP-rME83hQvAoryna1yrFbUDQHaJ5.jpeg', 'OIP-zdZAIubVm0NYpKeBu4zW5QHaFa.jpeg']
